# INTRODUCTION : Définition des constantes

In [1]:
# À décommenter uniquement lors de la première installation
# %pip install -r ../requirements.txt

In [2]:
import copy
import json
import random
import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from setfit import SetFitModel, Trainer, TrainingArguments




f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

DATA_DIR = ROOT / "Data"
SRC_DIR = ROOT / "SRC"
SPLIT_DIR = DATA_DIR / "SPLIT"
MODELS_DIR = ROOT / "Models"
EMBEDDING_DIR = DATA_DIR / "Embedding"
RESULTS_DIR = ROOT / "Results"

for directory in [
    DATA_DIR,
    SRC_DIR,
    SPLIT_DIR,
    MODELS_DIR,
    EMBEDDING_DIR,
    RESULTS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

from SRC.preprocessing import (
    clean_tweet_LSTM,
    clean_tweet_SETFIT,
)

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CONFIG = {
    "test_size": 50_000,
    "validation_size": 25_000,
    "lstm_train_size": 300_000,
    "setfit_examples_per_class": 512,
    "max_vocab": 30_000,
    "embedding_dim": 200,
    "max_length_lstm": 40,
}

#print(f"Appareil utilisé : {DEVICE}")

# PREPARATION DES DONNEES

## Choix 1 : On part du DATASET initial et on le split en train, validation et test

In [5]:
DATASET_PATH = (
    DATA_DIR
    / "training.1600000.processed.noemoticon.csv"
)

column_names = [
    "target",
    "tweet_id",
    "date",
    "query",
    "user",
    "tweet",
]

df = pd.read_csv(
    DATASET_PATH,
    encoding="ISO-8859-1",
    names=column_names,
    usecols=["target", "tweet"],
)

df["label"] = df["target"].map({0: 0, 4: 1})
df = df[["tweet", "label"]]

display(df.head())
print(f"Dimensions : {df.shape}")

,tweet,label
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0
1,is upset that he can't update his Facebook by ...,0
2,@Kenichan I dived many times for the ball. Man...,0
3,my whole body feels itchy and like its on fire,0
4,"@nationwideclass no, it's not behaving at all....",0


Dimensions : (1600000, 2)


### Controle du dataset

In [6]:
print("Valeurs manquantes :")
display(df.isna().sum().rename("nombre"))

print("Répartition des classes :")
display(
    df["label"]
    .value_counts()
    .sort_index()
    .rename(index={0: "négatif", 1: "positif"})
    .rename("effectif")
)

Valeurs manquantes :


tweet    0
label    0
Name: nombre, dtype: int64

Répartition des classes :


label
négatif    800000
positif    800000
Name: effectif, dtype: int64

### Création de splits

In [7]:
df_train_val, df_test = train_test_split(
    df,
    test_size=CONFIG["test_size"],
    random_state=SEED,
    stratify=df["label"],
)

df_train, df_val = train_test_split(
    df_train_val,
    test_size=CONFIG["validation_size"],
    random_state=SEED,
    stratify=df_train_val["label"],
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f"Train      : {len(df_train):,}")
print(f"Validation : {len(df_val):,}")
print(f"Test       : {len(df_test):,}")

Train      : 1,525,000
Validation : 25,000
Test       : 50,000


### Verification des splits

In [8]:
split_summary = pd.DataFrame(
    {
        "Jeu": ["Train", "Validation", "Test"],
        "Nombre d'observations": [
            len(df_train),
            len(df_val),
            len(df_test),
        ],
        "Part de positifs": [
            df_train["label"].mean(),
            df_val["label"].mean(),
            df_test["label"].mean(),
        ],
    }
)

display(
    split_summary.style.format(
        {
            "Nombre d'observations": "{:,.0f}",
            "Part de positifs": "{:.2%}",
        }
    )
)

,Jeu,Nombre d'observations,Part de positifs
0,Train,"1,525,000",50.00%
1,Validation,"25,000",50.00%
2,Test,"50,000",50.00%


### Sauvegarde des splits

In [9]:
df_train.to_csv(
    SPLIT_DIR / "train.csv",
    index=False,
)
df_val.to_csv(
    SPLIT_DIR / "val.csv",
    index=False,
)
df_test.to_csv(
    SPLIT_DIR / "test.csv",
    index=False,
)

print(f"Splits sauvegardés dans : {SPLIT_DIR}")

Splits sauvegardés dans : f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\Data\SPLIT


## Choix 2 : On a déjà créé et sauvegardé les splits, et on les charge tout simplement

In [ ]:
df_train = pd.read_csv(SPLIT_DIR / "train.csv")
df_val = pd.read_csv(SPLIT_DIR / "val.csv")
df_test = pd.read_csv(SPLIT_DIR / "test.csv")

print(f"Train      : {len(df_train):,}")
print(f"Validation : {len(df_val):,}")
print(f"Test       : {len(df_test):,}")

# PIPELINE 1 : Baseline LSTM

In [15]:
df_train_lstm = (
    df_train
    .sample(
        n=CONFIG["lstm_train_size"],
        random_state=SEED,
    )
    .reset_index(drop=True)
    .copy()
)

df_val_lstm = df_val.copy()
df_test_lstm = df_test.copy()

for dataframe in [
    df_train_lstm,
    df_val_lstm,
    df_test_lstm,
]:
    dataframe["text_lstm"] = (
        dataframe["tweet"]
        .fillna("")
        .astype(str)
        .apply(clean_tweet_LSTM)
    )

display(
    df_train_lstm[
        ["tweet", "text_lstm", "label"]
    ].head()
)

,tweet,text_lstm,label
0,Watching Ramsay's Kitchen Nightmares now Ew c...,watching ramsays kitchen nightmares now ew coc...,1
1,On my way too the beachh with thee bitchess a...,on my way too the beachh with thee bitchess ah...,1
2,"My hand is swollen, bruised and all the shit ...",my hand is swollen bruised and all the shit hu...,0
3,@MrBillyBones although it would be the highlig...,although it would be the highlight of the summ...,0
4,Going to the beach with Cody and Jake,going to the beach with cody and jake,1


In [16]:
tweet_lengths = (
    df_train_lstm["text_lstm"]
    .str.split()
    .str.len()
)

length_statistics = tweet_lengths.describe(
    percentiles=[0.90, 0.95, 0.99]
)

display(length_statistics.to_frame("longueur"))

print(
    "Longueur maximale retenue : "
    f"{CONFIG['max_length_lstm']} tokens"
)

,longueur
count,300000.000000
mean,12.480963
std,6.907725
min,0.000000
90%,23.000000
95%,25.000000
99%,28.000000
max,37.000000


Longueur maximale retenue : 40 tokens


## Construction du vocabulaire

In [17]:
counter = Counter()

for text in df_train_lstm["text_lstm"]:
    counter.update(text.split())

itos = ["<pad>", "<unk>"] + [
    word
    for word, _ in counter.most_common(
        CONFIG["max_vocab"] - 2
    )
]

stoi = {
    word: index
    for index, word in enumerate(itos)
}

PAD_INDEX = stoi["<pad>"]
UNK_INDEX = stoi["<unk>"]

print(f"Taille du vocabulaire : {len(itos):,}")

Taille du vocabulaire : 30,000


In [18]:
def construire_matrice_glove(
    stoi,
    glove_path,
    embedding_dim,
    seed=42,
):
    """Construit une matrice d'embeddings alignée sur le vocabulaire."""

    rng = np.random.default_rng(seed)

    embedding_matrix = rng.normal(
        loc=0,
        scale=0.1,
        size=(len(stoi), embedding_dim),
    ).astype(np.float32)

    embedding_matrix[PAD_INDEX] = 0.0

    words_found = 0

    with open(glove_path, encoding="utf-8") as file:
        for line in file:
            values = line.rstrip().split()
            word = values[0]
            word_index = stoi.get(word)

            if word_index is not None:
                embedding_matrix[word_index] = np.asarray(
                    values[1:],
                    dtype=np.float32,
                )
                words_found += 1

    return embedding_matrix, words_found

In [19]:
GLOVE_PATH = (
    EMBEDDING_DIR
    / "glove.twitter.27B.200d.txt"
)

EMBEDDING_MATRIX_PATH = (
    EMBEDDING_DIR
    / "emb_matrix_300k.npy"
)

if EMBEDDING_MATRIX_PATH.exists():
    embedding_matrix = np.load(
        EMBEDDING_MATRIX_PATH
    )
    print("Matrice GloVe existante rechargée.")

else:
    embedding_matrix, words_found = (
        construire_matrice_glove(
            stoi=stoi,
            glove_path=GLOVE_PATH,
            embedding_dim=CONFIG["embedding_dim"],
            seed=SEED,
        )
    )

    np.save(
        EMBEDDING_MATRIX_PATH,
        embedding_matrix,
    )

    coverage = words_found / len(stoi)

    print(
        f"Couverture GloVe : "
        f"{words_found:,}/{len(stoi):,} "
        f"({coverage:.1%})"
    )

print(
    f"Dimensions de la matrice : "
    f"{embedding_matrix.shape}"
)

Matrice GloVe existante rechargée.
Dimensions de la matrice : (30000, 200)


## Encodage 

In [20]:
def encoder_textes_lstm(texts):
    """Convertit une série de textes en matrice d'indices."""

    encoded_texts = np.full(
        shape=(
            len(texts),
            CONFIG["max_length_lstm"],
        ),
        fill_value=PAD_INDEX,
        dtype=np.int64,
    )

    for row_index, text in enumerate(texts):
        tokens = str(text).split()[
            :CONFIG["max_length_lstm"]
        ]

        encoded_texts[
            row_index,
            :len(tokens),
        ] = [
            stoi.get(token, UNK_INDEX)
            for token in tokens
        ]

    return torch.from_numpy(encoded_texts)

In [21]:
X_train_lstm = encoder_textes_lstm(
    df_train_lstm["text_lstm"]
)
X_val_lstm = encoder_textes_lstm(
    df_val_lstm["text_lstm"]
)
X_test_lstm = encoder_textes_lstm(
    df_test_lstm["text_lstm"]
)

y_train_lstm = torch.tensor(
    df_train_lstm["label"].to_numpy(),
    dtype=torch.float32,
)
y_val_lstm = torch.tensor(
    df_val_lstm["label"].to_numpy(),
    dtype=torch.float32,
)
y_test = df_test_lstm["label"].to_numpy()

print(f"Train      : {X_train_lstm.shape}")
print(f"Validation : {X_val_lstm.shape}")
print(f"Test       : {X_test_lstm.shape}")

Train      : torch.Size([300000, 40])
Validation : torch.Size([25000, 40])
Test       : torch.Size([50000, 40])


## Modèle BiLSTM

In [22]:
class BiLSTMClassifier(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        hidden_dim=128,
        dropout=0.3,
    ):
        super().__init__()

        vocabulary_size, embedding_dim = (
            embedding_matrix.shape
        )

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix),
            freeze=False,
            padding_idx=PAD_INDEX,
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            hidden_dim * 2,
            1,
        )

    def forward(self, inputs):
        embeddings = self.embedding(inputs)

        _, (hidden_state, _) = self.lstm(
            embeddings
        )

        representation = torch.cat(
            (
                hidden_state[-2],
                hidden_state[-1],
            ),
            dim=1,
        )

        logits = self.classifier(
            self.dropout(representation)
        )

        return logits.squeeze(1)

In [23]:
model_lstm = BiLSTMClassifier(
    embedding_matrix=embedding_matrix,
    hidden_dim=128,
    dropout=0.3,
).to(DEVICE)

print(model_lstm)

BiLSTMClassifier(
  (embedding): Embedding(30000, 200, padding_idx=0)
  (lstm): LSTM(200, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)


In [24]:
def predire_lstm(
    model,
    features,
    batch_size=512,
    threshold=0.5,
):
    """Produit les prédictions binaires du BiLSTM."""

    model.eval()
    predictions = []

    with torch.no_grad():
        for start in range(
            0,
            len(features),
            batch_size,
        ):
            batch = features[
                start:start + batch_size
            ].to(DEVICE)

            logits = model(batch)
            probabilities = torch.sigmoid(logits)

            batch_predictions = (
                probabilities >= threshold
            ).int()

            predictions.extend(
                batch_predictions.cpu().numpy()
            )

    return np.asarray(predictions)

In [25]:
LSTM_BATCH_SIZE = 256
LSTM_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model_lstm.parameters(),
    lr=1e-3,
)

best_validation_f1 = -np.inf
best_state = None
epochs_without_improvement = 0

lstm_history = []
training_start = time.time()

for epoch in range(1, LSTM_EPOCHS + 1):
    model_lstm.train()

    permutation = torch.randperm(
        len(X_train_lstm)
    )

    running_loss = 0.0
    batch_count = 0
    epoch_start = time.time()

    for start in range(
        0,
        len(X_train_lstm),
        LSTM_BATCH_SIZE,
    ):
        indices = permutation[
            start:start + LSTM_BATCH_SIZE
        ]

        features_batch = X_train_lstm[
            indices
        ].to(DEVICE)

        labels_batch = y_train_lstm[
            indices
        ].to(DEVICE)

        optimizer.zero_grad()

        logits = model_lstm(features_batch)
        loss = criterion(logits, labels_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        batch_count += 1

    validation_predictions = predire_lstm(
        model=model_lstm,
        features=X_val_lstm,
    )

    validation_labels = (
        y_val_lstm.numpy().astype(int)
    )

    validation_accuracy = accuracy_score(
        validation_labels,
        validation_predictions,
    )

    validation_f1 = f1_score(
        validation_labels,
        validation_predictions,
        average="macro",
    )

    mean_loss = running_loss / batch_count
    epoch_duration = time.time() - epoch_start

    lstm_history.append(
        {
            "epoch": epoch,
            "loss": mean_loss,
            "validation_accuracy": validation_accuracy,
            "validation_f1_macro": validation_f1,
            "duration_seconds": epoch_duration,
        }
    )

    print(
        f"Epoch {epoch}/{LSTM_EPOCHS} | "
        f"loss : {mean_loss:.4f} | "
        f"val accuracy : {validation_accuracy:.4f} | "
        f"val F1 macro : {validation_f1:.4f} | "
        f"durée : {epoch_duration:.0f} s"
    )

    if validation_f1 > best_validation_f1:
        best_validation_f1 = validation_f1
        best_state = copy.deepcopy(
            model_lstm.state_dict()
        )
        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print("Arrêt anticipé.")
            break

training_time_lstm = (
    time.time() - training_start
)

model_lstm.load_state_dict(best_state)

print(
    f"Meilleur F1 validation : "
    f"{best_validation_f1:.4f}"
)
print(
    f"Temps total : "
    f"{training_time_lstm:.1f} secondes"
)

Epoch 1/5 | loss : 0.4455 | val accuracy : 0.8144 | val F1 macro : 0.8143 | durée : 143 s
Epoch 2/5 | loss : 0.3851 | val accuracy : 0.8201 | val F1 macro : 0.8200 | durée : 173 s
Epoch 3/5 | loss : 0.3481 | val accuracy : 0.8189 | val F1 macro : 0.8189 | durée : 196 s
Epoch 4/5 | loss : 0.3109 | val accuracy : 0.8158 | val F1 macro : 0.8158 | durée : 185 s
Arrêt anticipé.
Meilleur F1 validation : 0.8200
Temps total : 697.7 secondes


In [26]:
lstm_history_df = pd.DataFrame(lstm_history)

display(
    lstm_history_df.style.format(
        {
            "loss": "{:.4f}",
            "validation_accuracy": "{:.4f}",
            "validation_f1_macro": "{:.4f}",
            "duration_seconds": "{:.1f}",
        }
    )
)

,epoch,loss,validation_accuracy,validation_f1_macro,duration_seconds
0,1,0.4455,0.8144,0.8143,143.3
1,2,0.3851,0.8201,0.8200,173.0
2,3,0.3481,0.8189,0.8189,195.9
3,4,0.3109,0.8158,0.8158,185.5


## Sauvegarde

In [27]:
LSTM_PATH = MODELS_DIR / "bilstm_glove.pt"

torch.save(
    {
        "state_dict": model_lstm.state_dict(),
        "itos": itos,
        "max_length": CONFIG["max_length_lstm"],
        "pad_index": PAD_INDEX,
        "unk_index": UNK_INDEX,
        "embedding_dim": CONFIG["embedding_dim"],
        "hidden_dim": 128,
        "bidirectional": True,
        "threshold": 0.5,
        "best_validation_f1": float(
            best_validation_f1
        ),
        "training_time_seconds": (
            training_time_lstm
        ),
    },
    LSTM_PATH,
)

print(f"BiLSTM sauvegardé dans : {LSTM_PATH}")

BiLSTM sauvegardé dans : f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\Models\bilstm_glove.pt


# PIPELINE 2 : Modèle SetFit

In [28]:
N_PER_CLASS = CONFIG[
    "setfit_examples_per_class"
]

df_train_setfit = (
    df_train
    .groupby("label", group_keys=False)
    .sample(
        n=N_PER_CLASS,
        random_state=SEED,
    )
    .reset_index(drop=True)
    .copy()
)

df_val_setfit = df_val.copy()
df_test_setfit = df_test.copy()

for dataframe in [
    df_train_setfit,
    df_val_setfit,
    df_test_setfit,
]:
    dataframe["text_setfit"] = (
        dataframe["tweet"]
        .fillna("")
        .astype(str)
        .apply(clean_tweet_SETFIT)
    )

print(
    f"Nombre d'exemples SetFit : "
    f"{len(df_train_setfit):,}"
)

display(
    df_train_setfit["label"]
    .value_counts()
    .sort_index()
)

Nombre d'exemples SetFit : 1,024


label
0    512
1    512
Name: count, dtype: int64

In [29]:
N_VALIDATION_PER_CLASS = 1_000

df_val_setfit_small = (
    df_val_setfit
    .groupby("label", group_keys=False)
    .sample(
        n=N_VALIDATION_PER_CLASS,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

print(
    f"Validation SetFit : "
    f"{len(df_val_setfit_small):,} tweets"
)

#df_val_setfit_small = df_val_setfit.copy()

Validation SetFit : 2,000 tweets


In [30]:
def convertir_en_dataset_setfit(
    dataframe,
):
    return Dataset.from_pandas(
        dataframe[
            ["text_setfit", "label"]
        ].rename(
            columns={"text_setfit": "text"}
        ),
        preserve_index=False,
    )


train_ds = convertir_en_dataset_setfit(
    df_train_setfit
)

val_ds = convertir_en_dataset_setfit(
    df_val_setfit_small
)

print(f"Train SetFit      : {len(train_ds):,}")
print(f"Validation SetFit : {len(val_ds):,}")

Train SetFit      : 1,024
Validation SetFit : 2,000


## Modèle SetFit

In [31]:
MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-mpnet-base-v2"
)

model_setfit = SetFitModel.from_pretrained(
    MODEL_NAME,
    labels=["négatif", "positif"],
)

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [32]:
def compute_metrics(y_pred, y_true):
    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
        ),
    }

In [33]:
setfit_args = TrainingArguments(
    output_dir=str(
        MODELS_DIR / "checkpoints_setfit"
    ),
    batch_size=(16, 16),
    num_epochs=(1, 8),
    num_iterations=10,
    body_learning_rate=2e-5,
    head_learning_rate=1e-2,
    max_length=64,
    seed=SEED,
    report_to="none",
    save_strategy="no",
)

trainer_setfit = Trainer(
    model=model_setfit,
    args=setfit_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    metric=compute_metrics,
)

Map: 100%|██████████| 1024/1024 [00:00<00:00, 16636.07 examples/s]


In [34]:
training_start = time.time()

trainer_setfit.train()

training_time_setfit = (
    time.time() - training_start
)

validation_metrics_setfit = (
    trainer_setfit.evaluate()
)

print(
    f"Temps d'entraînement SetFit : "
    f"{training_time_setfit:.1f} secondes"
)

print(validation_metrics_setfit)

***** Running training *****
  Num unique pairs = 20480
  Batch size = 16
  Num epochs = 1
f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.438100
50,0.282000
100,0.236000
150,0.191600
200,0.149700
250,0.109200
300,0.082500
350,0.074100
400,0.055300
450,0.037500


***** Running evaluation *****


Temps d'entraînement SetFit : 2582.0 secondes
{'accuracy': 0.823, 'f1_macro': 0.8229936277705998}


## Sauvegarde

In [35]:
SETFIT_DIR = (
    MODELS_DIR / "setfit_mpnet"
)

model_setfit.save_pretrained(
    str(SETFIT_DIR)
)

setfit_metadata = {
    "backbone": MODEL_NAME,
    "training_examples": len(
        df_train_setfit
    ),
    "examples_per_class": N_PER_CLASS,
    "training_time_seconds": (
        training_time_setfit
    ),
    "validation_metrics": (
        validation_metrics_setfit
    ),
}

with open(
    SETFIT_DIR / "metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        setfit_metadata,
        file,
        ensure_ascii=False,
        indent=4,
        default=float,
    )

print(f"SetFit sauvegardé dans : {SETFIT_DIR}")

SetFit sauvegardé dans : f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\Models\setfit_mpnet


# Comparaison finale

In [36]:
LABEL_MAPPING = {
    "négatif": 0,
    "positif": 1,
    "negative": 0,
    "positive": 1,
    "0": 0,
    "1": 1,
}


def convertir_predictions_setfit(
    predictions,
):
    predictions = np.asarray(predictions)

    if np.issubdtype(
        predictions.dtype,
        np.number,
    ):
        return predictions.astype(int)

    return np.asarray(
        [
            LABEL_MAPPING[str(prediction)]
            for prediction in predictions
        ],
        dtype=int,
    )

In [37]:
def predire_setfit_par_lots(
    model,
    texts,
    batch_size=256,
):
    predictions = []

    for start in range(
        0,
        len(texts),
        batch_size,
    ):
        batch = texts[
            start:start + batch_size
        ]

        predictions.extend(
            model.predict(batch)
        )

    return convertir_predictions_setfit(
        predictions
    )

In [38]:
def evaluer_predictions(
    y_true,
    y_pred,
    inference_time,
):
    return {
        "Accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "F1 macro": f1_score(
            y_true,
            y_pred,
            average="macro",
        ),
        "Temps d'inférence (s)": (
            inference_time
        ),
    }

In [39]:
start_time = time.time()

predictions_lstm = predire_lstm(
    model=model_lstm,
    features=X_test_lstm,
    batch_size=512,
    threshold=0.5,
)

inference_time_lstm = (
    time.time() - start_time
)

metrics_lstm_test = evaluer_predictions(
    y_true=y_test,
    y_pred=predictions_lstm,
    inference_time=inference_time_lstm,
)

print(metrics_lstm_test)

{'Accuracy': 0.81534, 'F1 macro': 0.8152697563248426, "Temps d'inférence (s)": 2.7415363788604736}


In [40]:
texts_test_setfit = (
    df_test_setfit["text_setfit"].tolist()
)

y_test_setfit = (
    df_test_setfit["label"].to_numpy()
)

start_time = time.time()

predictions_setfit = (
    predire_setfit_par_lots(
        model=model_setfit,
        texts=texts_test_setfit,
        batch_size=256,
    )
)

inference_time_setfit = (
    time.time() - start_time
)

metrics_setfit_test = evaluer_predictions(
    y_true=y_test_setfit,
    y_pred=predictions_setfit,
    inference_time=inference_time_setfit,
)

print(metrics_setfit_test)

{'Accuracy': 0.81742, 'F1 macro': 0.8174029608477333, "Temps d'inférence (s)": 338.95105481147766}


In [41]:
assert np.array_equal(
    y_test,
    y_test_setfit,
), "L'ordre des observations diffère entre les deux tests."

assert len(predictions_lstm) == len(y_test)
assert len(predictions_setfit) == len(y_test)

print(
    "Les deux modèles sont évalués sur "
    f"les mêmes {len(y_test):,} observations."
)

Les deux modèles sont évalués sur les mêmes 50,000 observations.


In [42]:
results_test = pd.DataFrame(
    [
        {
            "Modèle": "BiLSTM + GloVe",
            "Type": "Baseline",
            "Exemples d'entraînement": len(
                df_train_lstm
            ),
            "Accuracy": (
                metrics_lstm_test["Accuracy"]
            ),
            "F1 macro": (
                metrics_lstm_test["F1 macro"]
            ),
            "Temps d'entraînement (s)": (
                training_time_lstm
            ),
            "Temps d'inférence (s)": (
                inference_time_lstm
            ),
        },
        {
            "Modèle": "SetFit + MPNet",
            "Type": "Méthode récente",
            "Exemples d'entraînement": len(
                df_train_setfit
            ),
            "Accuracy": (
                metrics_setfit_test["Accuracy"]
            ),
            "F1 macro": (
                metrics_setfit_test["F1 macro"]
            ),
            "Temps d'entraînement (s)": (
                training_time_setfit
            ),
            "Temps d'inférence (s)": (
                inference_time_setfit
            ),
        },
    ]
)

display(
    results_test.style
    .format(
        {
            "Exemples d'entraînement": "{:,.0f}",
            "Accuracy": "{:.4f}",
            "F1 macro": "{:.4f}",
            "Temps d'entraînement (s)": "{:.1f}",
            "Temps d'inférence (s)": "{:.1f}",
        }
    )
    .highlight_max(
        subset=["Accuracy", "F1 macro"],
        color="lightgreen",
    )
    .highlight_min(
        subset=[
            "Exemples d'entraînement",
            "Temps d'entraînement (s)",
            "Temps d'inférence (s)",
        ],
        color="lightgreen",
    )
)

,Modèle,Type,Exemples d'entraînement,Accuracy,F1 macro,Temps d'entraînement (s),Temps d'inférence (s)
0,BiLSTM + GloVe,Baseline,"300,000",0.8153,0.8153,697.7,2.7
1,SetFit + MPNet,Méthode récente,"1,024",0.8174,0.8174,2582.0,339.0


In [43]:
RESULTS_PATH = (
    RESULTS_DIR / "comparaison_modeles.csv"
)

results_test.to_csv(
    RESULTS_PATH,
    index=False,
)

print(
    f"Résultats sauvegardés dans : "
    f"{RESULTS_PATH}"
)

Résultats sauvegardés dans : f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\Results\comparaison_modeles.csv


In [44]:
models_predictions = {
    "BiLSTM + GloVe": predictions_lstm,
    "SetFit + MPNet": predictions_setfit,
}

for model_name, predictions in (
    models_predictions.items()
):
    print(f"\n{'=' * 60}")
    print(model_name)
    print("=" * 60)

    print(
        classification_report(
            y_test,
            predictions,
            target_names=[
                "négatif",
                "positif",
            ],
            digits=4,
        )
    )


BiLSTM + GloVe
              precision    recall  f1-score   support

     négatif     0.8035    0.8348    0.8189     25000
     positif     0.8281    0.7958    0.8117     25000

    accuracy                         0.8153     50000
   macro avg     0.8158    0.8153    0.8153     50000
weighted avg     0.8158    0.8153    0.8153     50000


SetFit + MPNet
              precision    recall  f1-score   support

     négatif     0.8237    0.8078    0.8156     25000
     positif     0.8114    0.8271    0.8192     25000

    accuracy                         0.8174     50000
   macro avg     0.8175    0.8174    0.8174     50000
weighted avg     0.8175    0.8174    0.8174     50000



# Experience complémentaire

In [ ]:
N_VALUES = [64, 256, 512, 1000]

resultats_validation = []

# Même vérité terrain pour toutes les expériences
y_val_setfit = (
    df_val_setfit_small["label"]
    .to_numpy()
    .astype(int)
)

for n_per_class in N_VALUES:

    # Échantillon équilibré propre à chaque expérience
    train_sample = (
        df_train
        .groupby("label", group_keys=False)
        .sample(
            n=n_per_class,
            random_state=SEED,
        )
        .reset_index(drop=True)
        .copy()
    )

    train_sample["text_setfit"] = (
        train_sample["tweet"]
        .fillna("")
        .astype(str)
        .apply(clean_tweet_SETFIT)
    )

    train_ds_exp = Dataset.from_pandas(
        train_sample[
            ["text_setfit", "label"]
        ].rename(
            columns={"text_setfit": "text"}
        ),
        preserve_index=False,
    )

    model_exp = SetFitModel.from_pretrained(
        MODEL_NAME,
        labels=["négatif", "positif"],
    )

    trainer_exp = Trainer(
        model=model_exp,
        args=setfit_args,
        train_dataset=train_ds_exp,
        eval_dataset=val_ds,
        metric=compute_metrics,
    )

    start_time = time.time()

    trainer_exp.train()

    training_duration = time.time() - start_time

    predictions = predire_setfit_par_lots(
        model=model_exp,
        texts=df_val_setfit_small[
            "text_setfit"
        ].tolist(),
        batch_size=256,
    )

    resultats_validation.append(
        {
            "Exemples par classe": n_per_class,
            "Exemples totaux": 2 * n_per_class,
            "Accuracy validation": accuracy_score(
                y_val_setfit,
                predictions,
            ),
            "F1 macro validation": f1_score(
                y_val_setfit,
                predictions,
                average="macro",
            ),
            "Temps entraînement (s)": training_duration,
        }
    )

df_resultats_validation = pd.DataFrame(
    resultats_validation
)

display(
    df_resultats_validation.style.format(
        {
            "Accuracy validation": "{:.4f}",
            "F1 macro validation": "{:.4f}",
            "Temps entraînement (s)": "{:.1f}",
        }
    )
)

In [ ]:
data_ratio = (
    len(df_train_lstm)
    / len(df_train_setfit)
)

f1_difference = (
    metrics_setfit_test["F1 macro"]
    - metrics_lstm_test["F1 macro"]
)

print(
    f"SetFit utilise environ "
    f"{data_ratio:.1f} fois moins "
    f"d'exemples que le BiLSTM."
)

print(
    f"Différence de F1 macro "
    f"(SetFit - BiLSTM) : "
    f"{f1_difference:+.4f}"
)